In [11]:
import pandas as pd
import numpy as np

In [3]:
from epics import PV

ModuleNotFoundError: No module named 'epics'

In [ ]:
PCs = [PV('BME%dVV:setVolt' % (i+1)) for i in range(3)]
for PC in PCs:
    print('---------------')
    print(PC.info)
    print(PC.get())

In [10]:
detectornames = ['SCOPE1ZULP:h%dp%d:rdAmplAv' % (h+1, p+1) for h in range(2) for p in range(3)]
print(detectornames)

['SCOPE1ZULP:h1p1:rdAmplAv', 'SCOPE1ZULP:h1p2:rdAmplAv', 'SCOPE1ZULP:h1p3:rdAmplAv', 'SCOPE1ZULP:h2p1:rdAmplAv', 'SCOPE1ZULP:h2p2:rdAmplAv', 'SCOPE1ZULP:h2p3:rdAmplAv']


In [ ]:
detectoravgs    = [PV(name) for name in detectornames]
detectormonitor = PV('SCOPE1ZULP:h1p1:rdAmpl')
detectoravglen  = PV('SCOPE1ZULP:rdAvLength')

In [8]:
voltagestandard = [8350, 8800, 8100]

In [7]:
voltagestepsize = 250
voltageset = np.arange(0,9000+voltagestepsize, voltagestepsize)
print(voltageset)

[   0  250  500  750 1000 1250 1500 1750 2000 2250 2500 2750 3000 3250
 3500 3750 4000 4250 4500 4750 5000 5250 5500 5750 6000 6250 6500 6750
 7000 7250 7500 7750 8000 8250 8500 8750 9000]


In [ ]:
def callback_savevalues(**kwargs):
    count += 1
    if count >= averagelen_target and detectoravglen.get() >= averagelen_target:
        # we have collected enough data, save the current average and continue to the next point.
        outd = {'PC%d_setvolt' % i: PC.get() for i, PC in enumerate(PCs)}
        outd.update({detname: detavg.get() for detname, detavg in zip(detectornames, detectoravglen)})
        outd.update({'avglen': detectoravglen.get()})
        result_df = pd.concat([result_df, pd.DataFrame(outd, index=[0])], ignore_index = True)
        
        count = 0
        current_volt += 1
        if current_volt >= len(voltageset):
            # we are done with the voltage scan, go to the next PC
            current_volt = 0
            current_PC += 1
            for PC, standard in zip(PCs, voltagestandard):
                PC.put(standard) # all PCs to standard voltages
            if current_PC >= len(PCs):
                # we are done with all PCs and are finished!
                finished = True
                return
        PCs[current_PC].put(voltageset[current_volt]) # set current PC to the next voltage setpoint

In [4]:
count = 0
averagelen_target = 50
current_PC = 0
current_volt = 0
finished = False
result_df = pd.DataFrame()